Q1. Customer Satisfaction and Ordinal Encoding

(a) Apply OrdinalEncoder and verify rankings

In [1]:
import pandas as pd

In [2]:
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.DataFrame({"Customer_ID": range(101, 113),
    "Satisfaction": ["Poor", "Good", "Average", "Excellent","Poor", "Good", "Average", "Excellent",
    "Good", "Average", "Excellent", "Poor"],
    "Purchase_Amount": [500, 1200, 800, 2500, 400, 1500, 900, 3000, 1800, 700, 2800, 350],
    "Membership": ["Bronze", "Silver", "Bronze", "Gold", "Bronze", "Gold", "Silver", "Gold",
        "Silver", "Bronze", "Gold", "Silver"]})

satisfaction_order = ["Poor", "Average", "Good", "Excellent"]

membership_order = ["Bronze", "Silver", "Gold"]

sat_encoder = OrdinalEncoder(categories=[satisfaction_order])

mem_encoder = OrdinalEncoder(categories=[membership_order])

df["Satisfaction_Encoded"] = sat_encoder.fit_transform(
    df[["Satisfaction"]]
).astype(int)

df["Membership_Encoded"] = mem_encoder.fit_transform(
    df[["Membership"]]
).astype(int)

print(df)

sat_ranks = dict(zip(satisfaction_order, range(4)))
mem_ranks = dict(zip(membership_order, range(3)))

assert all(
    sat_ranks[k] < sat_ranks[v]
    for k, v in zip(satisfaction_order[:-1],
                    satisfaction_order[1:])
)

assert all(
    mem_ranks[k] < mem_ranks[v]
    for k, v in zip(membership_order[:-1],
                    membership_order[1:])
)

assert all(
    df["Satisfaction_Encoded"] ==
    df["Satisfaction"].map(sat_ranks)
)

assert all(
    df["Membership_Encoded"] ==
    df["Membership"].map(mem_ranks)
)

print("\nRanking verification successful!")


    Customer_ID Satisfaction  Purchase_Amount Membership  \
0           101         Poor              500     Bronze   
1           102         Good             1200     Silver   
2           103      Average              800     Bronze   
3           104    Excellent             2500       Gold   
4           105         Poor              400     Bronze   
5           106         Good             1500       Gold   
6           107      Average              900     Silver   
7           108    Excellent             3000       Gold   
8           109         Good             1800     Silver   
9           110      Average              700     Bronze   
10          111    Excellent             2800       Gold   
11          112         Poor              350     Silver   

    Satisfaction_Encoded  Membership_Encoded  
0                      0                   0  
1                      2                   1  
2                      1                   0  
3                      3       

(b) Calculate average purchase amount before and after encoding

In [4]:
avg_before = df.groupby("Satisfaction")["Purchase_Amount"].mean()

print("Average before encoding:")
print(avg_before)

avg_after = df.groupby("Satisfaction_Encoded")[
    "Purchase_Amount"
].mean()

print("\nAverage after encoding:")
print(avg_after)


Average before encoding:
Satisfaction
Average       800.000000
Excellent    2766.666667
Good         1500.000000
Poor          416.666667
Name: Purchase_Amount, dtype: float64

Average after encoding:
Satisfaction_Encoded
0     416.666667
1     800.000000
2    1500.000000
3    2766.666667
Name: Purchase_Amount, dtype: float64


Q2. Label Encoding vs. Ordinal Encoding in a Classification Problem

In [6]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

(a) Apply Ordinal Encoding and Label Encoding

In [7]:
df = pd.DataFrame({"Study_Level": ["Low", "Medium", "High", "Low", "High",
        "Medium", "High", "Low", "Medium", "High",
        "Low", "Medium"],
    "City": ["Bhopal", "Delhi", "Mumbai", "Delhi",
        "Bhopal", "Mumbai", "Delhi", "Mumbai",
        "Bhopal", "Delhi", "Mumbai", "Bhopal"],
    "Result": ["Fail", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Fail", "Pass"]})

oe = OrdinalEncoder(categories=[["Low", "Medium", "High"]])
df["Study_Encoded"] = oe.fit_transform(
    df[["Study_Level"]]
).astype(int)

le = LabelEncoder()
df["Result_Encoded"] = le.fit_transform(df["Result"])

city_map = {"Bhopal": 0,"Delhi": 1,"Mumbai": 2}
df["City_Label"] = df["City"].map(city_map)

print(df)

print("\nResult mapping:", dict(zip(
    le.classes_, le.transform(le.classes_)
)))

print("\nCity mapping:", city_map)


   Study_Level    City Result  Study_Encoded  Result_Encoded  City_Label
0          Low  Bhopal   Fail              0               0           0
1       Medium   Delhi   Pass              1               1           1
2         High  Mumbai   Pass              2               1           2
3          Low   Delhi   Fail              0               0           1
4         High  Bhopal   Pass              2               1           0
5       Medium  Mumbai   Pass              1               1           2
6         High   Delhi   Pass              2               1           1
7          Low  Mumbai   Fail              0               0           2
8       Medium  Bhopal   Pass              1               1           0
9         High   Delhi   Pass              2               1           1
10         Low  Mumbai   Fail              0               0           2
11      Medium  Bhopal   Pass              1               1           0

Result mapping: {'Fail': np.int64(0), 'Pass': np.i

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split

(b) Build a preprocessing workflow and train a classification model

In [10]:
from sklearn.preprocessing import (OrdinalEncoder, OneHotEncoder, LabelEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X = df[["Study_Level", "City"]]
y = df["Result"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

preprocessor = ColumnTransformer([
    ("study", OrdinalEncoder(categories=[["Low", "Medium", "High"]]), ["Study_Level"]),

    ("city", OneHotEncoder(
        handle_unknown="ignore"
    ), ["City"])
])

# Create ML pipeline
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded
)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Actual:", le.inverse_transform(y_test))
print("Predicted:", le.inverse_transform(y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=le.classes_,
    zero_division=0
))


Actual: ['Fail' 'Pass' 'Pass']
Predicted: ['Fail' 'Pass' 'Pass']
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         1
        Pass       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



Q3. Encoding Strategy for Mixed Categorical Data

(a) Identify ordinal and nominal columns and apply encoding

In [11]:
df = pd.DataFrame({"Size": ["Small", "Medium", "Large", "Medium", "Small", "Large", "XL", "Medium",
        "XL", "Small", "Large", "XL"],
    "Color": ["Red", "Blue", "Green", "Red","Green", "Blue", "Red", "Green",
        "Blue", "Red", "Blue", "Green"],
    "Satisfaction": ["Poor", "Good", "Average", "Excellent","Average", "Good", "Excellent", "Poor",
        "Good", "Average", "Excellent", "Good"],
    "Order_Value": [500, 900, 1500, 1100,650, 1700, 2200, 700,2100, 550, 1600, 2300]})

size_order = ["Small", "Medium", "Large", "XL"]
sat_order = ["Poor", "Average", "Good", "Excellent"]

encoder = OrdinalEncoder(
    categories=[size_order, sat_order]
)

df[["Size_Encoded", "Satisfaction_Encoded"]] = (
    encoder.fit_transform(df[["Size", "Satisfaction"]])
    .astype(int)
)

print(df)

size_map = {v: i for i, v in enumerate(size_order)}
sat_map = {v: i for i, v in enumerate(sat_order)}

assert (df["Size_Encoded"] == df["Size"].map(size_map)).all()
assert (df["Satisfaction_Encoded"] ==
        df["Satisfaction"].map(sat_map)).all()

print("\nEncoding verification successful!")


      Size  Color Satisfaction  Order_Value  Size_Encoded  \
0    Small    Red         Poor          500             0   
1   Medium   Blue         Good          900             1   
2    Large  Green      Average         1500             2   
3   Medium    Red    Excellent         1100             1   
4    Small  Green      Average          650             0   
5    Large   Blue         Good         1700             2   
6       XL    Red    Excellent         2200             3   
7   Medium  Green         Poor          700             1   
8       XL   Blue         Good         2100             3   
9    Small    Red      Average          550             0   
10   Large   Blue    Excellent         1600             2   
11      XL  Green         Good         2300             3   

    Satisfaction_Encoded  
0                      0  
1                      2  
2                      1  
3                      3  
4                      1  
5                      2  
6                

(b) Apply ColumnTransformer with mixed encoding

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal",
            OrdinalEncoder(
                categories=[
                    ["Small", "Medium", "Large", "XL"],
                    ["Poor", "Average", "Good", "Excellent"]
                ]
            ),
            ["Size", "Satisfaction"]
        ),
        (
            "nominal",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["Color"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

transformed = preprocessor.fit_transform(df)

encoded_df = pd.DataFrame(
    transformed,
    columns=preprocessor.get_feature_names_out()
)

print(encoded_df)


    Size  Satisfaction  Color_Blue  Color_Green  Color_Red  Order_Value  \
0    0.0           0.0         0.0          0.0        1.0        500.0   
1    1.0           2.0         1.0          0.0        0.0        900.0   
2    2.0           1.0         0.0          1.0        0.0       1500.0   
3    1.0           3.0         0.0          0.0        1.0       1100.0   
4    0.0           1.0         0.0          1.0        0.0        650.0   
5    2.0           2.0         1.0          0.0        0.0       1700.0   
6    3.0           3.0         0.0          0.0        1.0       2200.0   
7    1.0           0.0         0.0          1.0        0.0        700.0   
8    3.0           2.0         1.0          0.0        0.0       2100.0   
9    0.0           1.0         0.0          0.0        1.0        550.0   
10   2.0           3.0         1.0          0.0        0.0       1600.0   
11   3.0           2.0         0.0          1.0        0.0       2300.0   

    Size_Encoded  Satisf